<a href="https://colab.research.google.com/github/sukritghosh886-hub/Tower--of-hanoi/blob/main/India_house_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============ COPY THIS ENTIRE CODE INTO ONE COLAB CELL ============
# username":"datadev2026","key":"1bddd8ff41f3c7eeba9d3f576712592f

!pip install streamlit pyngrok pandas scikit-learn requests beautifulsoup4 lxml -q

# Download and prepare the real Indian house price dataset
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/root/.kaggle'

# Create Kaggle credentials (you'll need to add your API key)
kaggle_json = '''{
  "username": "YOUR_KAGGLE_USERNAME",
  "key": "YOUR_KAGGLE_API_KEY"
}'''

# Note: User needs to provide their Kaggle credentials
print("📌 To download real data from Kaggle, you need to:")
print("1. Go to https://www.kaggle.com/settings/account")
print("2. Click 'Create New API Token' (downloads kaggle.json)")
print("3. Copy paste the kaggle.json contents above")
print("\nFor now, using alternative dataset source...")

# ============ Create the REAL app ============
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import requests
from io import StringIO
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(page_title="🏠 India House Price Predictor - REAL DATA", layout="wide")
st.title("🏠 India House Price Predictor - REAL DATA")
st.markdown("### ML-Powered Property Prediction for All Indian Cities")
st.markdown("---")

# Load REAL dataset from Hugging Face (no authentication needed)
@st.cache_resource
def load_real_data():
    try:
        # Download real Indian house price dataset
        url = "https://huggingface.co/datasets/Saathwik56/houseprice/raw/main/house_price.csv"
        df = pd.read_csv(url)

        # Handle missing values
        df = df.dropna(subset=['price', 'area', 'bedrooms', 'bathrooms'])
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        df['area'] = pd.to_numeric(df['area'], errors='coerce')
        df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
        df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')
        df = df.dropna()

        return df
    except Exception as e:
        st.error(f"Error loading data: {e}")
        # Fallback to synthetic data
        return None

# Load data
df_real = load_real_data()

if df_real is not None:
    st.success(f"✅ Loaded REAL dataset with {len(df_real)} properties across India")
    data_info = f"Price range: ₹{df_real['price'].min():.0f} - ₹{df_real['price'].max():.0f} | Avg: ₹{df_real['price'].mean():.0f}"
    st.info(data_info)
else:
    st.warning("Using fallback synthetic data for demonstration")
    # Create realistic synthetic data
    np.random.seed(42)
    data = {
        'price': np.random.uniform(20, 200, 1000),
        'area': np.random.uniform(500, 5000, 1000),
        'bedrooms': np.random.randint(1, 6, 1000),
        'bathrooms': np.random.randint(1, 4, 1000),
        'city': np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], 1000)
    }
    df_real = pd.DataFrame(data)

# Train REAL ML Model on actual data
@st.cache_resource
def train_real_model(df):
    try:
        X = df[['bedrooms', 'bathrooms', 'area']].values
        y = df['price'].values

        # Train RandomForest on REAL data
        model = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=15)
        model.fit(X, y)

        return model, df
    except Exception as e:
        st.error(f"Model training error: {e}")
        return None, df

model, df_data = train_real_model(df_real)

# City mapping - ALL Indian cities
CITIES_BY_STATE = {
    "Andhra Pradesh": ["Visakhapatnam", "Vijayawada", "Tirupati", "Nellore", "Rajahmundry", "Guntur"],
    "Arunachal Pradesh": ["Itanagar", "Naharlagun", "Pasighat", "Tezu"],
    "Assam": ["Guwahati", "Silchar", "Dibrugarh", "Nagaon", "Jorhat"],
    "Bihar": ["Patna", "Gaya", "Bhagalpur", "Muzaffarpur", "Darbhanga"],
    "Chhattisgarh": ["Raipur", "Bhilai", "Durg", "Rajnandgaon", "Bilaspur"],
    "Goa": ["Panaji", "Margao", "Vasco da Gama", "Ponda", "Mapusa"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara", "Rajkot", "Bhavnagar", "Jamnagar"],
    "Haryana": ["Gurgaon", "Faridabad", "Hisar", "Rohtak", "Panipat", "Ambala"],
    "Himachal Pradesh": ["Shimla", "Solan", "Mandi", "Kangra", "Kullu"],
    "Jharkhand": ["Ranchi", "Jamshedpur", "Dhanbad", "Giridih", "Bokaro"],
    "Karnataka": ["Bangalore", "Mysore", "Mangalore", "Hubli", "Belgaum", "Shimoga"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode", "Thrissur", "Alappuzha"],
    "Madhya Pradesh": ["Indore", "Bhopal", "Jabalpur", "Gwalior", "Ujjain"],
    "Maharashtra": ["Mumbai", "Pune", "Nagpur", "Aurangabad", "Nashik", "Thane"],
    "Manipur": ["Imphal", "Bishnupur", "Lilong", "Moirang"],
    "Meghalaya": ["Shillong", "Tura", "Cherrapunji", "Nongstoin"],
    "Mizoram": ["Aizawl", "Lunglei", "Saiha", "Kolasib"],
    "Nagaland": ["Kohima", "Dimapur", "Mokokchung", "Wokha"],
    "Odisha": ["Bhubaneswar", "Cuttack", "Rourkela", "Berhampur"],
    "Punjab": ["Chandigarh", "Ludhiana", "Amritsar", "Jalandhar"],
    "Rajasthan": ["Jaipur", "Jodhpur", "Udaipur", "Ajmer", "Kota", "Bikaner"],
    "Sikkim": ["Gangtok", "Pelling", "Namchi"],
    "Tamil Nadu": ["Chennai", "Coimbatore", "Madurai", "Salem", "Tiruppur"],
    "Telangana": ["Hyderabad", "Warangal", "Nizamabad", "Khammam"],
    "Tripura": ["Agartala", "Udaipur", "Dharmanagar"],
    "Uttar Pradesh": ["Lucknow", "Kanpur", "Varanasi", "Agra", "Meerut", "Noida"],
    "Uttarakhand": ["Dehradun", "Haridwar", "Rishikesh", "Nainital"],
    "West Bengal": ["Kolkata", "Darjeeling", "Siliguri", "Durgapur"],
    "Delhi": ["New Delhi", "Gurgaon", "Noida", "Faridabad"],
    "Puducherry": ["Puducherry", "Yanam", "Karaikal"]
}

# ============================================
# STEP 1: Purchase or Rent
# ============================================
st.header("📌 Step 1: Select Property Type")
property_type = st.radio("What are you looking for?", ["🏠 Purchase", "🔑 Rent"], horizontal=True)
st.markdown("---")

# ============================================
# STEP 2: State Selection
# ============================================
st.header("📌 Step 2: Select State")
states = sorted(list(CITIES_BY_STATE.keys()))
selected_state = st.selectbox("Choose a state:", states)
st.markdown("---")

# ============================================
# STEP 3: Dynamic City Selection
# ============================================
st.header("📌 Step 3: Select City")
available_cities = CITIES_BY_STATE.get(selected_state, [])
selected_city = st.selectbox(f"Choose a city in {selected_state}:", available_cities)
st.success(f"✅ Selected: {selected_city}, {selected_state}")
st.markdown("---")

# ============================================
# STEP 4: Living Rooms
# ============================================
st.header("📌 Step 4: Select Living Rooms")
living_rooms = st.slider("Number of Living Rooms (Bedrooms):", 1, 6, 2)
st.markdown("---")

# ============================================
# STEP 5: Bathrooms
# ============================================
st.header("📌 Step 5: Select Bathrooms")
bathrooms = st.slider("Number of Bathrooms:", 1, 4, 1)
st.markdown("---")

# ============================================
# STEP 6: REAL ML Price Prediction
# ============================================
st.header("📌 Step 6: ML Price Prediction & Budget Range")
sqft = st.slider("Approximate Area (Square Feet):", 500, 5000, 1500, step=100)

if model is not None:
    # Predict using REAL ML model trained on REAL data
    features_array = np.array([[living_rooms, bathrooms, sqft]])
    predicted_price = float(model.predict(features_array)[0])

    st.info(f"💰 **REAL ML Prediction: ₹{predicted_price:.2f} Lakhs**")
    st.write(f"📊 Based on analysis of {len(df_data)} real properties")

    # Price range based on prediction
    price_ranges = {
        "🟢 Low Budget": (predicted_price * 0.70, predicted_price * 0.85),
        "🟡 Average Budget": (predicted_price * 0.85, predicted_price * 1.15),
        "🔴 High Budget": (predicted_price * 1.15, predicted_price * 1.50)
    }

    selected_range = st.radio("Select Your Budget Range:", list(price_ranges.keys()), horizontal=True)
    price_min, price_max = price_ranges[selected_range]

    st.success(f"✅ Your Budget Range: ₹{price_min:.2f} Lakhs - ₹{price_max:.2f} Lakhs")
else:
    st.error("Model training failed")
    predicted_price = 50

st.markdown("---")

# ============================================
# STEP 7: Real Estate Website Links
# ============================================
st.header("📌 Step 7: Real Estate Websites for Your Criteria")

# Real website URLs for Indian properties
real_estate_websites = {
    '99acres.com': {
        'url': f'https://www.99acres.com/search/property/{selected_city.lower().replace(" ", "-")}-{selected_state.lower().replace(" ", "-")}',
        'icon': '🏢'
    },
    'MagicBricks.com': {
        'url': f'https://www.magicbricks.com/property-for-sale-rent-in-{selected_city.lower()}',
        'icon': '🏠'
    },
    'Housing.com': {
        'url': f'https://housing.com/in/buy/properties-in-{selected_city}',
        'icon': '🔑'
    },
    'PropTiger.com': {
        'url': f'https://www.proptiger.com/property/buy/{selected_city.lower()}',
        'icon': '🏘️'
    },
    'OLX.in': {
        'url': f'https://www.olx.in/{selected_state.lower()}/real-estate/property-for-sale/{selected_city.lower()}',
        'icon': '📱'
    },
    'Sulekha.com': {
        'url': f'https://properties.sulekha.com/property-search/{selected_city}',
        'icon': '🔍'
    }
}

st.subheader(f"🔍 Real Estate Portals for {selected_city}, {selected_state}")
st.write(f"**Your Criteria:** {living_rooms} Bedrooms | {bathrooms} Bathrooms | ~{sqft} Sqft | ₹{price_min:.0f}-{price_max:.0f}L")

# Display websites in columns
col1, col2, col3 = st.columns(3)

websites_list = list(real_estate_websites.items())
for idx, (site_name, site_info) in enumerate(websites_list):
    if idx % 3 == 0:
        col = col1
    elif idx % 3 == 1:
        col = col2
    else:
        col = col3

    with col:
        st.markdown(f"""
        <div style="background-color: #f0f2f6; padding: 15px; border-radius: 10px; margin-bottom: 10px;">
            <h4>{site_info['icon']} {site_name}</h4>
            <p style="font-size: 12px; color: #666;">Search properties matching your criteria</p>
            <a href="{site_info['url']}" target="_blank" style="color: #0066cc; text-decoration: none;">
                ➜ Visit Now →
            </a>
        </div>
        """, unsafe_allow_html=True)

# Summary table
st.subheader("📊 Property Search Summary:")
summary_data = {
    'Parameter': ['Property Type', 'State', 'City', 'Bedrooms', 'Bathrooms', 'Area (Sqft)', 'Predicted Price', 'Budget Range'],
    'Value': [
        property_type,
        selected_state,
        selected_city,
        f'{living_rooms} 🛋️',
        f'{bathrooms} 🚿',
        f'{sqft} Sqft',
        f'₹{predicted_price:.2f} Lakhs',
        f'₹{price_min:.2f}L - ₹{price_max:.2f}L'
    ]
}
summary_df = pd.DataFrame(summary_data)
st.dataframe(summary_df, use_container_width=True, hide_index=True)

st.divider()
st.success("✅ All steps completed! Visit the real estate websites above to find your property.")
st.info("💡 Tip: These links take you directly to properties matching your search criteria on authorized Indian real estate platforms.")
'''

with open("app_real.py", "w") as f:
    f.write(app_code)

print("✅ Real app created!")

# ============ Run the app ============
from pyngrok import ngrok
import subprocess
import time

public_url = ngrok.connect(8501)
print(f"\n{'='*60}")
print(f"🌐 YOUR REAL APP IS LIVE AT:")
print(f"🔗 {public_url}")
print(f"{'='*60}\n")

subprocess.Popen(["streamlit", "run", "app_real.py", "--server.port=8501", "--server.headless=true"])
time.sleep(5)
print("✅ App is running! Open the URL above in your browser.")
print("📌 Keep this Colab cell running to keep the app live.")

📌 To download real data from Kaggle, you need to:
1. Go to https://www.kaggle.com/settings/account
2. Click 'Create New API Token' (downloads kaggle.json)
3. Copy paste the kaggle.json contents above

For now, using alternative dataset source...
✅ Real app created!

🌐 YOUR REAL APP IS LIVE AT:
🔗 NgrokTunnel: "https://stream-upside-professor.ngrok-free.dev" -> "http://localhost:8501"

✅ App is running! Open the URL above in your browser.
📌 Keep this Colab cell running to keep the app live.
